# Notebook 1 -
 Tools + a ReAct Agent (Hugging Face) for Medical Insurance Claims

This notebook is the **tool-calling fundamentals** notebook for the case study
described in `01_case_study_and_functions.md`. It builds:

1. A set of **`[TOOL]`** functions - plain, deterministic Python - for every
   step that does *not* need judgment (record lookups, code lookups, money
   math, document generation).
2. A minimal, hand-rolled **ReAct loop** (Thought → Action → Action Input →
   Observation → ... → Final Answer) powered by an **open-source Hugging
   Face model**, used *only* for the three steps that genuinely need
   reasoning:
   - `extract_diagnosis_and_treatment` (reading free-text doctor notes)
   - `check_diagnosis_treatment_consistency` (clinical judgment)
   - `check_eligibility` (interpreting policy rules/exclusions)

We build the ReAct loop by hand (no LangChain/AutoGen) so the mechanics are
fully visible: this is exactly what those frameworks do under the hood.

In [ ]:
import os
import json
import re

from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

## 1. Mock data

In a real system these would be database calls / API calls. Here they're
plain Python dictionaries so the notebook is fully self-contained and
reproducible.


In [ ]:
PATIENTS = {
    "P1042": {
        "patient_id": "P1042",
        "name": "R. Sharma",
        "insurance_id": "INS-778",
        "admission_date": "2026-08-20",
        "discharge_date": "2026-08-22",
        "days_admitted": 2,
    },
    "P2091": {
        "patient_id": "P2091",
        "name": "K. Verma",
        "insurance_id": "INS-501",
        "admission_date": "2026-08-18",
        "discharge_date": "2026-08-25",
        "days_admitted": 7,
    },
}

DOCTOR_NOTES = {
    "P1042": (
        "Patient presented with high fever and abdominal pain. Blood work "
        "confirmed typhoid fever. Started on IV ceftriaxone for 5 days, "
        "patient responded well and was discharged in stable condition."
    ),
    "P2091": (
        "Patient admitted with severe chest pain and shortness of breath. "
        "ECG and troponin confirmed acute myocardial infarction. Underwent "
        "emergency coronary angioplasty with stent placement. Recovery "
        "uneventful, discharged on dual antiplatelet therapy."
    ),
}

DOCTOR_SIGNOFF = {
    "P1042": {"signed_off": True, "signed_by": "Dr. A. Iyer",
               "timestamp": "2026-08-22T09:14:00"},
    "P2091": {"signed_off": True, "signed_by": "Dr. S. Rao",
               "timestamp": "2026-08-25T11:02:00"},
}

ICD_TABLE = {
    "typhoid fever": "A01.0",
    "acute myocardial infarction": "I21.9",
    "fracture of femur": "S72.9",
}

CPT_TABLE = {
    "iv ceftriaxone therapy": "J0696",
    "coronary angioplasty with stent placement": "92941",
    "physiotherapy session": "97110",
}

POLICIES = {
    "INS-778": {
        "policy_id": "INS-778",
        "coverage_limit": 50000,
        "room_rent_limit_per_day": 3000,
        "covered_procedures": ["J0696", "J0690", "99223"],
        "excluded_procedures": ["Z100"],
        "deductible": 1000,
    },
    "INS-501": {
        "policy_id": "INS-501",
        "coverage_limit": 300000,
        "room_rent_limit_per_day": 5000,
        "covered_procedures": ["92941", "99223"],
        "excluded_procedures": ["Z100"],
        "deductible": 5000,
    },
}

BILL_ITEMS = {
    "P1042": [
        {"item": "Room charges (2 days)", "category": "room", "amount": 6000},
        {"item": "IV ceftriaxone (5-day course)", "category": "medicine", "amount": 4500},
        {"item": "Lab tests", "category": "diagnostics", "amount": 2200},
        {"item": "Doctor consultation fees", "category": "professional", "amount": 1800},
    ],
    "P2091": [
        {"item": "Room / ICU charges (7 days)", "category": "room", "amount": 35000},
        {"item": "Coronary angioplasty + stent", "category": "procedure", "amount": 210000},
        {"item": "Lab & imaging", "category": "diagnostics", "amount": 18000},
        {"item": "Doctor consultation fees", "category": "professional", "amount": 9000},
    ],
}


## 2. `[TOOL]` functions - deterministic, no LLM involved

These are ordinary Python functions. They would be called directly by the
orchestrator notebook, and are also the functions the ReAct agent is
allowed to call as *actions* during its reasoning loop.


In [ ]:
# --- 2.1 Data retrieval -------------------------------------------------

def get_patient_record(patient_id: str) -> dict:
    """[TOOL] Return the admission record for a patient."""
    return PATIENTS[patient_id]

def get_doctor_notes(patient_id: str) -> str:
    """[TOOL] Return the raw free-text doctor's note for a patient."""
    return DOCTOR_NOTES[patient_id]

def get_doctor_signoff_status(patient_id: str) -> dict:
    """[TOOL] Return whether a doctor has signed off on this patient's record."""
    return DOCTOR_SIGNOFF[patient_id]

def get_insurance_policy(insurance_id: str) -> dict:
    """[TOOL] Return the policy terms for a given insurance ID."""
    return POLICIES[insurance_id]

def get_hospital_bill_items(patient_id: str) -> list:
    """[TOOL] Return itemized hospital charges for a patient."""
    return BILL_ITEMS[patient_id]


In [ ]:
# --- 2.2 Coding & validation --------------------------------------------

def lookup_icd_code(disease_name: str):
    """[TOOL] Look up the ICD-10 code for a disease name (case-insensitive)."""
    print(f"Diseaese Name : {disease_name.strip().lower()}")
    return ICD_TABLE.get(disease_name.strip().lower())

def lookup_procedure_code(treatment_name: str):
    """[TOOL] Look up the CPT-style procedure code for a treatment name."""
    return CPT_TABLE.get(treatment_name.strip().lower())

def validate_code_exists(code: str, code_type: str) -> bool:
    """[TOOL] Confirm a code exists in the relevant reference table."""
    table = ICD_TABLE if code_type.upper() == "ICD" else CPT_TABLE
    return code in table.values()


In [ ]:
# --- 2.3 Money math -------------------------------------------------------

def calculate_total_bill(bill_items: list) -> float:
    """[TOOL] Sum all itemized charges."""
    return float(sum(item["amount"] for item in bill_items))

def calculate_covered_amount(policy: dict, total_bill: float, days_admitted: int,
                              eligibility_result: dict) -> float:
    """[TOOL] Deterministically compute how much insurance will pay, once
    eligibility has already been decided."""
    if not eligibility_result.get("eligible"):
        return 0.0
    cap_from_eligibility = eligibility_result.get("eligible_amount_cap")
    cap_from_policy = policy["coverage_limit"] - policy["deductible"]
    room_cap = policy["room_rent_limit_per_day"] * days_admitted
    # Room charges are capped separately; everything else is capped by the
    # smaller of the eligibility cap and the overall policy cap.
    room_charge = next((i["amount"] for i in [3000, 2000] ), 0)  # todo, see note below
    binding_cap = min(c for c in [cap_from_eligibility, cap_from_policy] if c is not None)
    return float(min(total_bill, binding_cap))

def calculate_patient_payable(total_bill: float, covered_amount: float) -> float:
    """[TOOL] Whatever isn't covered, the patient pays."""
    return float(max(0.0, total_bill - covered_amount))


> **Todo:** `calculate_covered_amount` extend case study to apply the
> `room_rent_limit_per_day` cap to the room-charge line item *specifically*
> (by filtering `bill_items` for `category == "room"`), rather than only
> capping the grand total.


In [ ]:
# --- 2.4 Output generation & escalation -----------------------------------

def generate_claim_form(patient_record: dict, diagnosis_code: str, procedure_code: str,
                         covered_amount: float, eligibility_reasoning: str) -> dict:
    """[TOOL] Fill in a claim form template."""
    return {
        "claim_id": f"CLM-{patient_record['patient_id']}-{patient_record['discharge_date'].replace('-', '')}",
        "patient_name": patient_record["name"],
        "insurance_id": patient_record["insurance_id"],
        "diagnosis_code": diagnosis_code,
        "procedure_code": procedure_code,
        "covered_amount": covered_amount,
        "eligibility_reasoning": eligibility_reasoning,
        "status": "ready_to_submit",
    }

def generate_discharge_bill(patient_record: dict, total_bill: float,
                             covered_amount: float, patient_payable: float) -> dict:
    """[TOOL] Fill in the patient-facing discharge bill."""
    return {
        "bill_id": f"BILL-{patient_record['patient_id']}-{patient_record['discharge_date'].replace('-', '')}",
        "patient_name": patient_record["name"],
        "total_bill": total_bill,
        "covered_amount": covered_amount,
        "amount_due": patient_payable,
    }

def flag_for_manual_review(patient_id: str, reason: str) -> dict:
    """[TOOL] Record that a human needs to look at this claim."""
    return {"patient_id": patient_id, "status": "manual_review", "reason": reason}


## What would be the expected changes if manual_review is needed?

## 3. The tool registry

The ReAct agent needs (a) a callable Python function per tool and (b) a
plain-language description + parameter schema, so the LLM knows the tool
exists and how to call it. This mirrors the "tools" list you'd pass to any
function-calling API (Hugging Face's chat-template tool calling).


In [ ]:
TOOL_REGISTRY = {
    "lookup_icd_code": {
        "fn": lookup_icd_code,
        "description": "Look up the ICD-10 code for a disease name.",
        "parameters": {"disease_name": "string"},
    },
    "lookup_procedure_code": {
        "fn": lookup_procedure_code,
        "description": "Look up the CPT-style procedure code for a treatment name.",
        "parameters": {"treatment_name": "string"},
    },
    "validate_code_exists": {
        "fn": validate_code_exists,
        "description": "Check whether a code exists in the ICD or CPT reference table.",
        "parameters": {"code": "string", "code_type": "'ICD' or 'CPT'"},
    },
}

def render_tool_schemas(tool_names):
    """Render a compact text description of the allowed tools for the prompt."""
    lines = []
    for name in tool_names:
        spec = TOOL_REGISTRY[name]
        params = ", ".join(f"{k}: {v}" for k, v in spec["parameters"].items())
        lines.append(f"- {name}({params}) -- {spec['description']}")
    return "\n".join(lines)

print(render_tool_schemas(["lookup_icd_code", "lookup_procedure_code", "validate_code_exists"]))


## What is the use of TOOL-REGISTRY?

## 4. Loading an open-source Hugging Face model

We use a small, freely available instruct model so this notebook is runnable
on modest hardware. Swap the
`MODEL_NAME` for anything bigger if you have the compute:

- `Qwen/Qwen2.5-1.5B-Instruct` (default here - small, good instruction
  following)
- `microsoft/Phi-3-mini-4k-instruct` (a bit larger, strong reasoning)
- `NousResearch/Hermes-2-Pro-Mistral-7B` (supports native tool-calling chat
  templates, if you want to try Section 7's bonus approach)

If you don't have local GPU access, use the Hugging Face **Inference API**
via `huggingface_hub.InferenceClient`


In [ ]:
# 4a. Local model via transformers -----------------------------------------
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto" if torch.cuda.is_available() else None,
# )

# def call_llm(prompt: str, max_new_tokens: int = 300, stop_strings=None) -> str:
#     """Send a raw prompt to the local HF model and return only the newly
#     generated text (the completion, not the echoed prompt)."""
#     messages = [{"role": "user", "content": prompt}]
#     inputs = tokenizer.apply_chat_template(
#         messages, add_generation_prompt=True, return_tensors="pt"
#     )
#     if torch.cuda.is_available():
#         inputs = inputs.to(model.device)
#     output_ids = model.generate(
#         inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         temperature=None,
#         top_p=None,
#         pad_token_id=tokenizer.eos_token_id,
#     )
#     completion = tokenizer.decode(
#         output_ids[0][inputs.shape[-1]:], skip_special_tokens=True
#     )
#     if stop_strings:
#         for s in stop_strings:
#             idx = completion.find(s)
#             if idx != -1:
#                 completion = completion[: idx + len(s)]
#     return completion

#
# Uncomment this block and comment out 4b if you'd download weights locally.

In [ ]:
# 4b. Hugging Face Inference API (no local download needed) ---


client = InferenceClient(model="Qwen/Qwen2.5-1.5B-Instruct", token=hf_token)

def call_llm(prompt: str, max_new_tokens: int = 300, stop_strings=None) -> str:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    completion = response.choices[0].message.content
    if stop_strings:
        for s in stop_strings:
            idx = completion.find(s)
            if idx != -1:
                completion = completion[: idx + len(s)]
    return completion

## Does swapping the model makes any difference in the generated answer?

## 5. A hand-rolled ReAct loop

This is the core teaching artifact. We give the model:
- The task it needs to accomplish
- The tools it's allowed to call, with schemas
- Strict formatting instructions so we can parse its output

Then we loop: call the LLM → parse `Action` / `Action Input` → run the real
Python function → append the result as `Observation` → repeat, until the
model emits `Final Answer:`.


In [ ]:
REACT_SYSTEM_TEMPLATE = """You are a careful reasoning agent solving a task \
step by step using the ReAct pattern.

You have access to these tools:
{tool_schemas}

Use exactly this format, one step at a time:

Thought: <your reasoning about what to do next>
Action: <one tool name from the list above>
Action Input: <a JSON object with the tool's arguments>

After you receive an "Observation:" for that action, continue with another
Thought/Action/Action Input, or, once you have enough information, write:

Thought: <final reasoning>
Final Answer: <a JSON object with your final structured answer>

Only ever output ONE Thought/Action/Action Input block, or ONE
Thought/Final Answer block, per turn. Never invent an Observation yourself.

Task:
{task}
"""

def parse_react_step(text: str):
    """Pull out (action, action_input_dict) or ('FINAL', answer_dict) from
    one turn of model output."""
    if "Final Answer:" in text:
        answer_text = text.split("Final Answer:", 1)[1].strip()
        try:
            return "FINAL", json.loads(answer_text)
        except json.JSONDecodeError:
            return "FINAL", {"raw": answer_text}
    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match = re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if not action_match or not input_match:
        return None, None
    action = action_match.group(1).strip()
    try:
        action_input = json.loads(input_match.group(1))
    except json.JSONDecodeError:
        action_input = {}
    return action, action_input


def run_react_agent(task: str, allowed_tools: list, max_steps: int = 5, verbose: bool = True):
    """Run a ReAct loop, restricted to `allowed_tools`, until Final Answer
    or max_steps is reached."""
    tool_schemas = render_tool_schemas(allowed_tools)
    prompt = REACT_SYSTEM_TEMPLATE.format(tool_schemas=tool_schemas, task=task)
    transcript = prompt
    print(f"PROMPT: {prompt}")

    for step in range(max_steps):
        completion = call_llm(transcript, stop_strings=["Observation:"])
        if verbose:
            print(completion.strip())

        kind, payload = parse_react_step(completion)

        if kind == "FINAL":
            return payload

        if kind is None or payload is None:
            # Model didn't follow the format -- nudge it and stop for safety.
            return {"error": "could_not_parse_step", "raw": completion}

        action, action_input = kind, payload
        if action not in allowed_tools:
            observation = f"ERROR: tool '{action}' is not in the allowed tool list."
        else:
            try:
                observation = TOOL_REGISTRY[action]["fn"](**action_input)
            except Exception as e:
                observation = f"ERROR calling {action}: {e}"

        obs_text = f"\nObservation: {observation}\n"
        if verbose:
            print(obs_text)
        transcript += "\n" + completion.strip() + obs_text

    return {"error": "max_steps_reached"}


## 6. Wrapping the three `[AGENT]` steps

Each of these is a thin wrapper: it builds a task description (including the
relevant unstructured text or data) and calls `run_react_agent` with the
tools that step is allowed to use. This is the pattern to generalize: **an
agent step = a task description + a restricted toolset + the ReAct loop.**


In [ ]:
def extract_diagnosis_and_treatment(doctor_notes_text: str) -> dict:
    """[AGENT] Extract a structured diagnosis + treatment from free text,
    self-checking against the ICD/CPT tables."""
    task = f"""Read the following doctor's note and extract the diagnosis and
the treatment given, in plain clinical language (not codes). Then confirm
each maps to a valid code using the tools available, retrying with a more
standard phrasing if a lookup returns null. Give your Final Answer as JSON:
{{"diagnosis": "...", "treatment": "...", "icd_code": "...", "cpt_code": "..."}}

Doctor's note:
\"\"\"{doctor_notes_text}\"\"\"
"""
    return run_react_agent(task, allowed_tools=["lookup_icd_code", "lookup_procedure_code"])


def check_diagnosis_treatment_consistency(diagnosis: str, treatment: str) -> dict:
    """[AGENT] Judge whether a treatment is clinically reasonable for a
    diagnosis. No tools needed here -- pure LLM judgment -- so we allow an
    empty toolset and expect an immediate Final Answer."""
    task = f"""Judge whether the following treatment is a clinically
reasonable choice for the stated diagnosis. Give your Final Answer as JSON:
{{"consistent": true/false, "reasoning": "...", "needs_human_review": true/false}}

Diagnosis: {diagnosis}
Treatment: {treatment}
"""
    return run_react_agent(task, allowed_tools=[])


def check_eligibility(policy: dict, diagnosis_code: str, procedure_code: str,
                       days_admitted: int) -> dict:
    """[AGENT] Interpret policy coverage/exclusion lists and caps to decide
    eligibility and the eligible amount cap."""
    task = f"""Given this insurance policy and this claim's codes, decide
whether the claim is eligible, and if so, compute the eligible amount cap
(coverage_limit minus deductible, unless an exclusion applies). Use the
validate_code_exists tool if you need to double check a code is a real CPT
code before deciding. Give your Final Answer as JSON:
{{"eligible": true/false, "eligible_amount_cap": number or null, "reasoning": "..."}}

Policy: {json.dumps(policy)}
Diagnosis code: {diagnosis_code}
Procedure code: {procedure_code}
Days admitted: {days_admitted}
"""
    return run_react_agent(task, allowed_tools=["validate_code_exists"])


## Why the above above functionalities are defined as Agents?

## 7. Try it out

Run each agent-wrapped function on the sample patient and inspect the full
ReAct trace printed to the console.


In [ ]:
notes = get_doctor_notes("P1042")
extraction_result = extract_diagnosis_and_treatment(notes)
extraction_result


In [ ]:
consistency_result = check_diagnosis_treatment_consistency(
    diagnosis="Typhoid fever",
    treatment="IV ceftriaxone therapy (5 days)",
)
consistency_result


In [ ]:
policy = get_insurance_policy("INS-778")
eligibility_result = check_eligibility(
    policy=policy,
    diagnosis_code="A01.0",
    procedure_code="J0696",
    days_admitted=2,
)
eligibility_result


## 8. Which steps were tools, which needed the agent?

| Step | Type | Why |
|---|---|---|
| `get_patient_record`, `get_doctor_notes`, `get_doctor_signoff_status`, `get_insurance_policy`, `get_hospital_bill_items` | **[TOOL]** | Plain lookups. No ambiguity, no natural language to interpret. |
| `lookup_icd_code`, `lookup_procedure_code`, `validate_code_exists` | **[TOOL]** | Deterministic table lookups. |
| `extract_diagnosis_and_treatment` | **[AGENT]** | Input is unstructured, free-text clinical prose; needs language understanding. |
| `check_diagnosis_treatment_consistency` | **[AGENT]** | Requires clinical judgment — a rule table can't cover every diagnosis/treatment pair. |
| `check_eligibility` | **[AGENT]** | Requires interpreting a policy's covered/excluded lists and caps together — genuinely ambiguous at the edges (e.g. partial exclusions, combination therapies). |
| `calculate_total_bill`, `calculate_covered_amount`, `calculate_patient_payable` | **[TOOL]** | Pure arithmetic, once eligibility is known. |
| `generate_claim_form`, `generate_discharge_bill` | **[TOOL]** | Template filling. |
| `flag_for_manual_review` | **[TOOL]** | A record-keeping side effect, invoked *by* the agent as an action when it isn't confident. |

## 9. Exercises

1. Extend `check_diagnosis_treatment_consistency` to also call
   `flag_for_manual_review` (add it to `allowed_tools`) whenever it decides
   `needs_human_review: true`, instead of just returning that flag.
2. Try the "messier" doctor's note from `02_react_transcript.md`
   ("gave standard IV antibiotics", no drug named) through
   `extract_diagnosis_and_treatment` and see whether the model asks for
   review instead of guessing.
3. Swap `MODEL_NAME` for `NousResearch/Hermes-2-Pro-Mistral-7B` and use its
   native tool-calling chat template (`tokenizer.apply_chat_template(...,
   tools=[...])`) instead of the hand-rolled text parser in Section 5.
   Compare reliability.
4. Add a fourth `[TOOL]`, `check_room_rent_cap(bill_items, policy,
   days_admitted)`, and refactor `calculate_covered_amount` to call it,
   fixing the placeholder noted in Section 2.3.
